# Validate data in silver zone

In [1]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F, types as T
from dotenv import load_dotenv
import os

load_dotenv('../../../../.env_file/infra.env'), load_dotenv('../../../../.env_file/app.env')

(True, True)

In [2]:
spark = (
    SparkSession.builder
    .appName('ingest data to silver zone')
    # delta lake support
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    # read/write S3
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY_ID'))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_ACCESS_KEY'))
    # jar files supporting read/write postgres, trino, S3, hive (delta table for lake house)
    .config(
        'spark.jars',
        """
            ../../../../spark/jar/trino-jdbc-480.jar,
            ../../../../spark/jar/hadoop-aws-3.4.2.jar,
            ../../../../spark/jar/bundle-2.29.52.jar,
            ../../../../spark/jar/delta-spark_4.1_2.13-4.2.0.jar,
            ../../../../spark/jar/delta-kernel-api-4.2.0.jar,
            ../../../../spark/jar/delta-kernel-defaults-4.2.0.jar,
            ../../../../spark/jar/delta-storage-4.2.0.jar,
        """
    )
    # Hive metastore
    .config("spark.hadoop.hive.metastore.uris", "thrift://localhost:9084")
    .config("spark.sql.catalogImplementation", "hive")
    .enableHiveSupport()
    .getOrCreate()
)

spark

26/06/22 16:48:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
spark.sql('create database silver location "s3a://flezi-data/silver/"')

AnalysisException: [SCHEMA_ALREADY_EXISTS] Cannot create schema `silver` because it already exists.
Choose a different name, drop the existing schema, or add the IF NOT EXISTS clause to tolerate pre-existing schema. SQLSTATE: 42P06

In [ ]:
df = spark.read.table("taxi_data_silver.nyc_taxi")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [6]:
df.show()

26/06/22 16:50:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-------------------+-------------------+---------------+-----------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+----+-----+
|           vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|    trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|year|month|
+--------------------+-------------------+-------------------+---------------+-----------------+------------+------------------+------------------+---------------+-------------

In [12]:
# Connection variables
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""

# Must CAST all timestamp(3) with time zone columns — SELECT * still exposes original types
query = """(
    SELECT 
        vendor_id,
        CAST(pickup_datetime AS timestamp) AS pickup_datetime,
        CAST(dropoff_datetime AS timestamp) AS dropoff_datetime,
        passenger_count, trip_distance, rate_code_id, store_and_fwd_flag,
        pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude,
        pu_location_id, do_location_id, payment_type,
        fare_amount, extra, mta_tax, surcharge, improvement_surcharge,
        tip_amount, tolls_amount, total_amount, congestion_surcharge,
        airport_fee, cbd_congestion_fee, year, month
    FROM lakehouse.taxi_data_silver.nyc_taxi
) AS t"""

df = spark.read.format("jdbc") \
    .option("url", trino_url) \
    .option("driver", "io.trino.jdbc.TrinoDriver") \
    .option("query", query) \
    .option("user", trino_user) \
    .option("password", trino_password) \
    .load()

df.show()

+--------------------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+------------------+----------------+--------------+--------------+------------+-----------------+-----+-------+---------+---------------------+----------+------------+-----------------+--------------------+-----------+------------------+----+-----+
|           vendor_id|    pickup_datetime|   dropoff_datetime|passenger_count|     trip_distance|rate_code_id|store_and_fwd_flag|  pickup_longitude|pickup_latitude| dropoff_longitude|dropoff_latitude|pu_location_id|do_location_id|payment_type|      fare_amount|extra|mta_tax|surcharge|improvement_surcharge|tip_amount|tolls_amount|     total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|year|month|
+--------------------+-------------------+-------------------+---------------+------------------+------------+------------------+------------------+---------------+----------

In [4]:
# Connection variables
trino_url = "jdbc:trino://localhost:8082"
trino_user = "trino"
trino_password = ""
target_table = "lakehouse.taxi_data_silver.nyc_taxi"

# Read table into a DataFrame
df = spark.read \
    .option("pushDownPredicate", "false") \
    .format("jdbc") \
    .option("url", trino_url) \
    .option("driver", "io.trino.jdbc.TrinoDriver") \
    .option("dbtable", target_table) \
    .option("user", trino_user) \
    .option("password", trino_password) \
    .load()

df.show()

Py4JJavaError: An error occurred while calling o66.load.
: org.apache.spark.SparkSQLException: [UNRECOGNIZED_SQL_TYPE] Unrecognized SQL type - name: timestamp(3) with time zone, id: TIMESTAMP_WITH_TIMEZONE. SQLSTATE: 42704
	at org.apache.spark.sql.errors.QueryExecutionErrors$.unrecognizedSqlTypeError(QueryExecutionErrors.scala:1032)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.getCatalystType(JdbcUtils.scala:245)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.$anonfun$getSchema$1(JdbcUtils.scala:330)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.sql.execution.datasources.jdbc.JdbcUtils$.getSchema(JdbcUtils.scala:330)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.$anonfun$getQueryOutputSchema$3(JDBCRDD.scala:91)
	at scala.util.Using$.resource(Using.scala:296)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCRDD$.$anonfun$getQueryOutputSchema$2(JDBCRDD.scala:89)
	at scala.util.Using$.resource(Using.scala:296)
	at